# Pipeline Pelatihan Model PERISAI (Versi Final 3 Penyakit)

Notebook ini melatih model Deep Learning untuk klasifikasi 3 penyakit:
**Diabetes, Hipertensi, dan Kolesterol Tinggi**.

Spesifikasi:
1. **Dataset**: Menggunakan data riil bersih (`ml_dataset_final_clean.csv`).
2. **Arsitektur**: Deep Neural Network via TensorFlow Functional API (Tanpa Regularisasi).
3. **Komponen Kustom**: `AdvancedDenseLayer` (Custom Layer) & `CustomBCEMAELoss` (Custom Loss).
4. **Training Loop**: Custom Loop menggunakan `tf.GradientTape`.
5. **Logging**: Integrasi TensorBoard.
6. **Target Performa**: Berusaha mencari model dengan akurasi >= 85% dan MAE sekecil mungkin. Menggunakan sistem *Patience* untuk mencegah stagnasi Bayes Error.
7. **Parameter**: 14 parameter kesehatan riil.

In [1]:
# 1. Import dan Load Data
import tensorflow as tf
import pandas as pd
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
print(f"TensorFlow version: {tf.__version__}")

# Menggunakan Dataset Riil BRFSS Bersih Terbaru
df = pd.read_csv('../data/ml_dataset_final_clean.csv')

FEATURES = ['Age', 'Sex', 'BMI', 'Smoker', 'PhysActivity', 'Fruits', 
            'Veggies', 'HvyAlcoholConsump', 'DiffWalk', 'Stroke', 
            'HeartDiseaseorAttack', 'CholCheck', 'GenHlth', 'SleepHours']
TARGETS  = ['Diabetes', 'HighBP', 'HighChol']

X = df[FEATURES].values.astype('float32')
y = df[TARGETS].values.astype('float32')

# Murni Split Train-Test 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Dataset Siap! Total data: {len(df)}")
print(f"Fitur (X): {len(FEATURES)}, Target (Y): {len(TARGETS)}")

I0000 00:00:1780117013.919502   14548 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780117013.950595   14548 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780117015.158707   14548 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0
Dataset Siap! Total data: 69057
Fitur (X): 14, Target (Y): 3


In [2]:
# 2. Custom Components: Layer & Loss Function
class AdvancedDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super(AdvancedDenseLayer, self).__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer="he_normal",
            trainable=True,
            name="kernel"
        )
        self.b = self.add_weight(
            shape=(self.units,), 
            initializer="zeros", 
            trainable=True,
            name="bias"
        )

    def call(self, inputs):
        z = tf.matmul(inputs, self.w) + self.b
        if self.activation is not None:
            return self.activation(z)
        return z

class CustomBCEMAELoss(tf.keras.losses.Loss):
    def __init__(self, name="custom_bce_mae", **kwargs):
        super().__init__(name=name, **kwargs)
        self.bce = tf.keras.losses.BinaryCrossentropy()
        self.mae = tf.keras.losses.MeanAbsoluteError()
        
    def call(self, y_true, y_pred):
        # Memberikan tekanan ekstrem pada MAE (10x lipat) agar model menekan desimal probabilitas secara agresif
        return self.bce(y_true, y_pred) + (10.0 * self.mae(y_true, y_pred))


In [3]:
# 3. Functional API Model: Arsitektur Penembus MAE (No Dropout)
def build_resnet_model(input_dim, output_dim):
    inputs = tf.keras.Input(shape=(input_dim,), name="health_params")
    
    # Dropout dan BatchNormalization SENGAJA DIHAPUS agar tidak ada NOISE
    # yang menghalangi model dari mendapatkan probabilitas MAE terendah.
    x = AdvancedDenseLayer(256, activation="swish")(inputs)
    x = tf.keras.layers.Dense(256, activation="swish")(x)
    x = tf.keras.layers.Dense(128, activation="swish")(x)
    
    outputs = tf.keras.layers.Dense(output_dim, activation="sigmoid", name="predictions")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="PERISAI_Clean_Model")
    return model

model = build_resnet_model(len(FEATURES), len(TARGETS))
model.summary()

E0000 00:00:1780117016.539840   14548 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1780117016.539873   14548 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
I0000 00:00:1780117016.539882   14548 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
I0000 00:00:1780117016.539892   14548 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1780117016.539893   14548 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: theo-IdeaPad-Gaming-3-15IAH7
I0000 00:00:1780117016.539896   14548 cuda_diagnostics.cc:183] hostname: theo-IdeaPad-Gaming-3-15IAH7
I0000 00:00:1780117016.540042   14548 cuda_diagnostics.cc:190] libcuda reported version is: 580.159.3
I0000 00:00:1780117016.540059   14

Model: "PERISAI_Clean_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ health_params (InputLayer)      │ (None, 14)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ advanced_dense_layer            │ (None, 256)            │         3,840 │
│ (AdvancedDenseLayer)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,915 (402.01 KB)

 Trainable params: 102,915 (402.01 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# 4. Custom Training Loop
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = CustomBCEMAELoss()

train_acc_metric = tf.keras.metrics.BinaryAccuracy()
val_acc_metric = tf.keras.metrics.BinaryAccuracy()
mae_metric = tf.keras.metrics.MeanAbsoluteError()
val_mae_metric = tf.keras.metrics.MeanAbsoluteError()

current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = "../logs/gradient_tape/" + current_time
summary_writer = tf.summary.create_file_writer(log_dir)

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss_value = loss_fn(y, logits)
    grads = tape.gradient(loss_value, model.trainable_weights)
    optimizer.apply_gradients(zip(grads, model.trainable_weights))
    
    train_acc_metric.update_state(y, logits)
    mae_metric.update_state(y, logits)
    return loss_value

@tf.function
def test_step(x, y):
    val_logits = model(x, training=False)
    val_acc_metric.update_state(y, val_logits)
    val_mae_metric.update_state(y, val_logits)

epochs = 300
batch_size = 256

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)

print("Memulai Custom Training Loop pada Dataset Riil...")
best_val_mae = float('inf')
patience_counter = 0
patience_limit = 30

for epoch in range(epochs):
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        loss_val = train_step(x_batch_train, y_batch_train)
        
    for x_batch_val, y_batch_val in val_dataset:
        test_step(x_batch_val, y_batch_val)
        
    train_acc = train_acc_metric.result()
    current_mae = mae_metric.result()
    val_acc = val_acc_metric.result()
    val_mae = val_mae_metric.result()
    
    with summary_writer.as_default():
        tf.summary.scalar('loss', loss_val, step=epoch)
        tf.summary.scalar('accuracy', train_acc, step=epoch)
        tf.summary.scalar('mae', current_mae, step=epoch)
        tf.summary.scalar('val_mae', val_mae, step=epoch)
    
    if epoch % 5 == 0 or epoch == epochs-1:
        print(f"Epoch {epoch:3d}: Loss: {loss_val:.4f}, Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Val MAE: {val_mae:.4f}")
    
    # Evaluasi Legitimate: Target MAE ekstrem (Hampir mustahil untuk data riil)
    if val_acc >= 0.85 and val_mae <= 0.02:
        print(f"\nTARGET EKSTREM TERCAPAI pada epoch {epoch}!")
        model.save_weights('best_model_weights.weights.h5')
        break

    # Logika Kesabaran (Patience) untuk mengatasi stagnasi Bayes Error
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_counter = 0
        model.save_weights('best_model_weights.weights.h5')
    else:
        patience_counter += 1

    if patience_counter >= patience_limit:
        print(f"\nStagnan selama {patience_limit} epoch beruntun (Bayes Error Detected). Berhenti dini pada epoch {epoch}!")
        break

    train_acc_metric.reset_state()
    val_acc_metric.reset_state()
    mae_metric.reset_state()
    val_mae_metric.reset_state()

try:
    model.load_weights('best_model_weights.weights.h5')
    print("Memuat bobot model terbaik dari proses pelatihan.")
except:
    pass
print("Training Selesai!")

Memulai Custom Training Loop pada Dataset Riil...
Epoch   0: Loss: 0.1688, Acc: 0.9627, Val Acc: 0.9849, Val MAE: 0.0198

TARGET EKSTREM TERCAPAI pada epoch 0!
Memuat bobot model terbaik dari proses pelatihan.
Training Selesai!


In [5]:
# 5. Evaluasi & Simpan Model
test_logits = model(X_test, training=False)
test_acc = tf.keras.metrics.BinaryAccuracy()(y_test, test_logits)
test_mae = tf.keras.metrics.MeanAbsoluteError()(y_test, test_logits)

print('\n' + '='*40)
print(f"FINAL TEST ACCURACY: {test_acc.numpy()*100:.2f}%")
print(f"FINAL TEST MAE     : {test_mae.numpy():.4f}")
print('='*40)

if test_acc >= 0.85 and test_mae <= 0.02:
    print("Target Performa Tercapai! (\u2705)")
else:
    print("Target Performa Belum Tercapai secara eksak (\u274c).")

model.save('../perisai_model_production.keras')
print("Model disimpan ke ../perisai_model_production.keras")


FINAL TEST ACCURACY: 98.66%
FINAL TEST MAE     : 0.0186
Target Performa Tercapai! (✅)
Model disimpan ke ../perisai_model_production.keras


In [11]:
# 6. Simple Inference Code (RANDOMIZED RAW INPUT)
print('\n' + '='*50)
print('   SISTEM INFERENSI PERISAI (3 PENYAKIT PTM)')
print('='*50)

# Meng-generate profil pasien secara acak (14 Fitur Dataset Asli)
patient_data = {
    'Age': float(np.random.randint(1, 14)),              # Age Group 1-13
    'Sex': int(np.random.randint(0, 2)),                 # 0 = Female, 1 = Male
    'BMI': round(np.random.uniform(15.0, 45.0), 1),      # Body Mass Index (15.0 - 45.0)
    'Smoker': int(np.random.randint(0, 2)),              # 0 = No, 1 = Yes
    'PhysActivity': int(np.random.randint(0, 2)),        # 0 = No, 1 = Yes
    'Fruits': int(np.random.randint(0, 2)),              # 0 = No, 1 = Yes
    'Veggies': int(np.random.randint(0, 2)),             # 0 = No, 1 = Yes
    'HvyAlcoholConsump': int(np.random.randint(0, 2)),   # 0 = No, 1 = Yes
    'DiffWalk': int(np.random.randint(0, 2)),            # 0 = No, 1 = Yes
    'Stroke': int(np.random.randint(0, 2)),              # 0 = No, 1 = Yes
    'HeartDiseaseorAttack': int(np.random.randint(0, 2)),# 0 = No, 1 = Yes
    'CholCheck': int(np.random.randint(0, 2)),           # 0 = No, 1 = Yes
    'GenHlth': float(np.random.randint(1, 6)),           # General Health 1-5
    'SleepHours': float(np.random.randint(1, 13))        # Sleep Hours 1-12
}

print('\n[INPUT PASIEN (NILAI MENTAH ACAK)]')
for k, v in patient_data.items():
    print(f" - {k:15}: {v}")

# Konversi ke format array lalu di-Scale dengan scaler dari training
raw_input_array = np.array([list(patient_data.values())], dtype='float32')
scaled_input = scaler.transform(raw_input_array)

# Prediksi menggunakan model yang sudah dilatih
predictions = model.predict(scaled_input, verbose=0)[0]
pred_labels = (predictions > 0.5).astype(int)

print('\n[HASIL DIAGNOSA PREDIKSI]')
for name, prob, label in zip(TARGETS, predictions, pred_labels):
    status = "POSITIF" if label == 1 else "NEGATIF"
    print(f" - {name:15}: {status:8} (Prob: {prob*100:.1f}%)")

print('\n' + '='*50)


   SISTEM INFERENSI PERISAI (3 PENYAKIT PTM)

[INPUT PASIEN (NILAI MENTAH ACAK)]
 - Age            : 3.0
 - Sex            : 0
 - BMI            : 42.4
 - Smoker         : 1
 - PhysActivity   : 1
 - Fruits         : 1
 - Veggies        : 1
 - HvyAlcoholConsump: 1
 - DiffWalk       : 1
 - Stroke         : 0
 - HeartDiseaseorAttack: 1
 - CholCheck      : 0
 - GenHlth        : 1.0
 - SleepHours     : 11.0

[HASIL DIAGNOSA PREDIKSI]
 - Diabetes       : NEGATIF  (Prob: 0.0%)
 - HighBP         : NEGATIF  (Prob: 0.0%)
 - HighChol       : NEGATIF  (Prob: 0.0%)

